In [1]:
from pathlib import Path

from IPython.display import HTML, Markdown, display

import altair as alt
import polars as pl

from shantay.metadata import Metadata
from shantay.schema import KEYWORDS_MINOR_PROTECTION

ROOT = Path("/Volumes/dsa-sor-db/data")
TIMELINE_WIDTH = 1_000

frame = pl.DataFrame(
    Metadata.read_json(ROOT).records()
).with_columns(
    pl.col("release").str.to_date("%Y-%m-%d"),
    (pl.col("batch_rows") / pl.col("total_rows") * 100).alias("protection_of_minors_pct"),
    (pl.col("batch_rows_with_keywords") / pl.col("batch_rows") * 100).alias("batch_keywords_pct"),
    (pl.col("total_rows_with_keywords") / pl.col("total_rows") * 100).alias("total_keywords_pct"),
)

start_date, stop_date, batch_rows, batch_memory, batch_pct, total_rows = (
    frame.select(
        pl.col("release").min().alias("min"),
        pl.col("release").max().alias("max"),
        pl.col("batch_rows").sum(),
        pl.col("batch_memory").sum(),
        pl.col("protection_of_minors_pct").mean(),
        pl.col("total_rows").sum(),
    ).row(0)
)
batch_keywords_pct, total_keywords_pct = (
    frame.select(
        pl.col("batch_keywords_pct").mean(),
        pl.col("total_keywords_pct").mean(),
    ).row(0)
)

if 1_000_000_000 < batch_memory:
    batch_memory = batch_memory / 1_000_000_000
    unit = "GB"
else:
    batch_memory = batch_memory / 1_000_000
    unit = "MB"

display(HTML("""<h1>EU DSA SoR Category: Protection of Minors</h1>
<dl>
<dt><strong>EU</strong></dt>
<dd><strong>European Union</strong>, a club of European countries</dd>

<dt><strong>DSA</strong></dt>
<dd><strong>Digital Services Act</strong>, an EU law regulating digital service
providers, with compliance burden dependent on service size</dd>

<dt><strong>SoR</strong></dt>
<dd><strong>Statement of Reason</strong>, an explanation for a content
moderation action by a service provider</dd>

</dl>
"""))

summary = {
    "Dates":
        f"__{start_date} to {stop_date}__ (inclusive)",
    "Rows with _Protection of Minors_ category":
        f"{batch_rows:,} out of {total_rows:,}, i.e., __{batch_pct:.1f}% of all rows__",
    "Rows with _Protection of Minors_ category and keywords":
        f"__{batch_keywords_pct:.1f}% of rows in category__ "
        f"(versus {total_keywords_pct:.1f}% across all categories)",
    "Total bytes for in-memory frames":
        f"__{batch_memory:,.1f} {unit}__",
}
widths = [max(len(k) for k in summary.keys()), max(len(v) for v in summary.values())]
summary_md = (
    f"|{'Description':<{widths[0]}}|{'Value':<{widths[1]}}|\n"
    + f"|{'-' * widths[0]}|{'-' * widths[1]}|\n"
    + "".join([f"|{k:<{widths[0]}}|{v:<{widths[1]}}|\n" for k, v in summary.items()])
)

schema_md = (
    f"|{'Column':<30}|{'Type':<20}|\n|{'-' * 30}|{'-' * 20}|\n"
    + "".join([f"|{str(key):<30}|{str(value):<20}|\n" for key, value in frame.schema.items()])
)

display(Markdown(f"""
### Summary
{summary_md}

### Schema
{schema_md}

"""))


### Summary
|Description                                           |Value                                                           |
|------------------------------------------------------|----------------------------------------------------------------|
|Dates                                                 |__2023-09-25 to 2025-01-31__ (inclusive)                        |
|Rows with _Protection of Minors_ category             |54,948,370 out of 25,396,757,168, i.e., __0.3% of all rows__    |
|Rows with _Protection of Minors_ category and keywords|__9.5% of rows in category__ (versus 3.5% across all categories)|
|Total bytes for in-memory frames                      |__46.2 GB__                                                     |


### Schema
|Column                        |Type                |
|------------------------------|--------------------|
|release                       |Date                |
|batch_count                   |Int64               |
|total_rows                    |Int64               |
|total_rows_with_keywords      |Int64               |
|batch_rows                    |Int64               |
|batch_rows_with_keywords      |Int64               |
|batch_memory                  |Int64               |
|protection_of_minors_pct      |Float64             |
|batch_keywords_pct            |Float64             |
|total_keywords_pct            |Float64             |




In [2]:
alt.theme.enable("default")
COLOR_PALETTE = [
    "#4269D0",
    "#EFB118",
    "#FF725C",
    "#6CC5B0",
    "#3CA951",
    "#FF8AB7",
    "#A463F2",
    "#97BBF5",
    "#9C6B4E",
    "#9498A0",
]

In [3]:
# Timeline #1
f1 = frame.select(
    pl.col("release"),
    pl.col("batch_rows") / 1_000,
    pl.col("batch_memory") / 1_000_000,
)

timeline1 = (alt
    .Chart(
        f1,
        title="Statements of Reason: Protection of Minors — Daily Counts",
    ).mark_bar(
        tooltip=True,
        color="#6CC5B0",
    ).encode(
        alt.X("release:T"),
        alt.Y("batch_rows:Q").title("thousand rows"),
    ).properties(
        width=TIMELINE_WIDTH,
    ).interactive()
)

In [4]:
# Timeline #2
f2 = frame.select(
    pl.col("release"),
    (pl.col("batch_rows") / pl.col("total_rows") * 100).alias("protection_of_minors")
)

timeline2 = (
    alt.Chart(
        f2,
        title="Statements of Reason: Protection of Minors — Daily Percentage",
    ).mark_bar(
        tooltip=True,
        color="#6CC5B0",
    ).encode(
        alt.X("release:T"),
        alt.Y("protection_of_minors:Q").title("percent"),
    )
    .properties(width=TIMELINE_WIDTH)
    .interactive()
)

In [5]:
# Timeline #3
f3 = frame.select(
    pl.col("release"),
    pl.col("batch_keywords_pct").alias("Protection of Minors Only"),
    pl.col("total_keywords_pct").alias("All SoRs"),
).unpivot(
    index=["release"],
    on=["Protection of Minors Only", "All SoRs"],
    variable_name="kind",
    value_name="pct",
)

timeline3 = (
    alt.Chart(
        f3,
        title="Statements of Reason: With Keywords — Daily Percentage",
    ).mark_line(
        tooltip=True,
    ).encode(
        alt.X("release:T"),
        alt.Y("pct:Q").title("percent"),
        alt.Color("kind:N").scale(
            domain=["Protection of Minors Only", "All SoRs"], range=["#FF8AB7", "#4269D0"]
        ),
        #order=alt.Order("kind", sort="ascending")
    ).properties(
        width=TIMELINE_WIDTH,
    )
    .interactive()
)

In [6]:
# Timeline #4
monthly = pl.read_parquet(ROOT / "monthly.parquet")

f4 = monthly.select(
    pl.col("date", "platforms_with_keywords", "platforms")
).rename({
    "date": "date",
    "platforms_with_keywords": "Platforms w/ Keywords",
    "platforms": "All Platforms",
}).unpivot(
    index=["date"],
    on=["Platforms w/ Keywords", "All Platforms"],
    variable_name="kind",
    value_name="count",
)

timeline4 = (
    alt.Chart(f4, title="Protection of Minors SoRs: Platforms — Monthly Counts").mark_line(tooltip=True).encode(
        alt.X("date:T"),
        alt.Y("count:Q"),
        alt.Color("kind:N").scale(
            domain=["Platforms w/ Keywords", "All Platforms"], range=["#FF725C", "#9498A0"],
        ),
    )
    .properties(width=TIMELINE_WIDTH)
    .interactive()
)

In [7]:
# Timeline #5
KEYWORDS = [kw.lower() for kw in KEYWORDS_MINOR_PROTECTION if kw != "NO_KEYWORD"]
MAPPING = {
    "keyword_age_specific_restrictions_minors": "Restricted",
    "keyword_child_sexual_abuse_material": "CSAM",
    "keyword_child_sexual_abuse_material_deepfake": "Deepfakes",
    "keyword_grooming_sexual_enticement_minors": "Grooming",
    "keyword_other": "Other",
    "keyword_unsafe_challenges": "Unsafe Challenges",
}
MAPPED_KEYWORDS = ["Restricted", "CSAM", "Deepfakes", "Grooming", "Other", "Unsafe Challenges"]

f5 = monthly.select(
    pl.col("date", *KEYWORDS)
).rename(
    MAPPING
).unpivot(
    index=["date"],
    on=MAPPED_KEYWORDS,
    variable_name="keyword",
    value_name="count",
)

timeline5 = (
    alt.Chart(f5, title="Protection of Minors SoRs: Keywords — Monthly Counts").mark_bar(
        tooltip=True,
        size=45,
        width=alt.RelativeBandSize(0.9),
        #width={"band": 200},
    ).encode(
        alt.X("date:T"),
        alt.Y("count:Q"),
        alt.Color("keyword:N").scale(
            domain=["CSAM", "Deepfakes", "Grooming", "Other", "Restricted", "Unsafe Challenges"],
            range=["#EFB118", "#6CC5B0", "#FF725C", "#A463F2", "#97BBF5", "#FF8AB7"],
        ),
    )
    .properties(width=TIMELINE_WIDTH)
    .interactive()
)

In [8]:
display(HTML("<h1>Statements of Reason Through Time</h1>"))
timelines = [timeline1, timeline2, timeline3, timeline4, timeline5]
graph = alt.vconcat(*timelines).resolve_scale(
    x="shared",  # Make sure that all the timelines are aligned
    color="independent",
).configure_range(
    category={"scheme": COLOR_PALETTE},
)
graph

alt.VConcatChart(...)